# Planificacion automatica aplicada al Senku
## Experimentacion - Convocatoria de junio

Este cuaderno acompana al sistema desarrollado en `senku/src`. Sigue la metodologia de la Practica 4 de la asignatura: se utiliza la biblioteca `unified_planning` para parsear y representar problemas PDDL y `OneshotPlanner` con `Fast Downward` como referencia. Sobre esa misma infraestructura ejecutamos nuestras propias implementaciones de **BFS** (parte comun) y **Beam Search** con la **funcion pagoda** (algoritmo especifico de la convocatoria de junio).

Las variantes 1, 3 y 5 son las exigidas por el enunciado; las 2 y 4 se incluyen para experimentacion adicional.

In [1]:
import sys
from pathlib import Path

# Permite ejecutar el notebook desde senku/notebooks/ sin instalar el paquete
RAIZ = Path.cwd().parent.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from senku.src.tableros import TABLEROS, dibuja_tablero, VARIANTES_OBLIGATORIAS
from senku.src.estado import ProblemaSenku
from senku.src.heuristicas import (
    pagoda_clasica, pagoda_uniforme,
    heuristica_pagoda, heuristica_compuesta, heuristica_conectividad, valor_pagoda,
)
from senku.src.busqueda import (
    busqueda_primero_anchura,
    beam_search,
    beam_search_con_reinicios,
)
from senku.src.dominio_up import construye_problema_up
from senku.src.lector_pddl import carga_problema_pddl, carga_con_unified_planning
from senku.src.planificador import resuelve_con_fast_downward

print(f'Variantes definidas: {list(TABLEROS.keys())}')
print(f'Obligatorias (junio): {VARIANTES_OBLIGATORIAS}')

Variantes definidas: [1, 2, 3, 4, 5]
Obligatorias (junio): (1, 3, 5)


## 1. Inspeccion de los tableros

Visualizamos los cinco tableros con su estado inicial (`o` = casilla ocupada, `.` = hueco).

In [2]:
for numero, tablero in TABLEROS.items():
    obligatoria = ' (obligatoria)' if numero in VARIANTES_OBLIGATORIAS else ''
    print(f'\n=== Variante {numero}{obligatoria}: {tablero.nombre} ({len(tablero.casillas)} casillas) ===')
    print(dibuja_tablero(tablero, tablero.inicial_ocupadas))


=== Variante 1 (obligatoria): variante_1_cruz_inglesa (33 casillas) ===
    o o o    
    o o o    
o o o o o o o
o o o . o o o
o o o o o o o
    o o o    
    o o o    

=== Variante 2: variante_2_cuadrado_5x5 (25 casillas) ===
o o o o o
o o o o o
o o . o o
o o o o o
o o o o o

=== Variante 3 (obligatoria): variante_3_octagonal_europeo (37 casillas) ===
    o o o    
  o o o o o  
o o o o o o o
o o o . o o o
o o o o o o o
  o o o o o  
    o o o    

=== Variante 4: variante_4_diamante (25 casillas) ===
      o      
    o o o    
  o o o o o  
o o o . o o o
  o o o o o  
    o o o    
      o      

=== Variante 5 (obligatoria): variante_5_cruz_extendida (45 casillas) ===
      o o o      
      o o o      
      o o o      
o o o o o o o o o
o o o o . o o o o
o o o o o o o o o
      o o o      
      o o o      
      o o o      


## 2. Pagoda de los estados iniciales

Comprobamos que la asignacion clasica de pagoda cumple la cota `a + b >= c` para todas las ternas de salto y calculamos la pagoda inicial y meta. Si no hubiera violaciones, podemos confiar en que la heuristica `h_pagoda` es admisible para esa variante.

In [3]:
def valida_pagoda(pesos, problema):
    return [(d, s, h) for d, s, h in problema.saltos if pesos[d] + pesos[s] < pesos[h]]

for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero)
    pesos = pagoda_clasica(tablero)
    pag_ini = valor_pagoda(p.inicial, pesos)
    pag_meta = sum(pesos[c] for c in p.meta_ocupadas if c in pesos)
    print(f'V{numero}: violaciones={len(valida_pagoda(pesos, p))} | '
          f'pagoda inicial={pag_ini} | pagoda meta={pag_meta} | exceso={pag_ini - pag_meta}')

V1: violaciones=0 | pagoda inicial=188 | pagoda meta=8 | exceso=180
V2: violaciones=0 | pagoda inicial=152 | pagoda meta=8 | exceso=144
V3: violaciones=0 | pagoda inicial=212 | pagoda meta=8 | exceso=204
V4: violaciones=0 | pagoda inicial=148 | pagoda meta=8 | exceso=140
V5: violaciones=0 | pagoda inicial=236 | pagoda meta=8 | exceso=228


## 3. Lectura del problema desde PDDL (estilo Practica 4)

Validamos el requisito de la convocatoria: el sistema recibe dos ficheros .pddl y los procesa con `unified_planning`. Los detalles del backend estan en `senku/src/lector_pddl.py`.

In [4]:
ruta_dominio = RAIZ / 'senku' / 'pddl' / 'dominio_senku.pddl'
for numero in [1, 3, 5]:
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_con_unified_planning(ruta_dominio, ruta_problema)
    print(f'V{numero}: {p.tablero.nombre} -> {len(p.tablero.casillas)} casillas, '
          f'{len(p.inicial)} piezas iniciales, {len(p.saltos)} saltos posibles')

V1: variante_1_cruz_inglesa -> 33 casillas, 32 piezas iniciales, 76 saltos posibles
V3: variante_3_octagonal_europeo -> 37 casillas, 36 piezas iniciales, 92 saltos posibles
V5: variante_5_cruz_extendida -> 45 casillas, 44 piezas iniciales, 108 saltos posibles


## 4. Linea base: Fast Downward via unified-planning

Antes de evaluar nuestro Beam Search, comparamos contra Fast Downward (el planificador recomendado en la Practica 4). Esto nos permite saber, para cada variante, si la instancia es resoluble en absoluto y cuanto cuesta encontrar el plan optimo.

Si Fast Downward marca una variante como `UNSOLVABLE_INCOMPLETELY`, significa que ha demostrado por refutacion completa que no existe plan (por ejemplo, las variantes 2 y 4 son irresolubles por paridad).

In [ ]:
# Llamada directa a Fast Downward via unified-planning.
#
# Notas:
#   - En Windows, ProcessPoolExecutor no puede serializar funciones
#     definidas en el notebook (BrokenProcessPool al hacer .result()).
#     Usamos el timeout interno del propio planificador.
#   - up-fast-downward 0.5.2 tiene un bug que dispara UnicodeDecodeError
#     al decodificar la salida de FD cuando contiene caracteres no-ASCII.
#     Lo neutralizamos con un parche tolerante (senku/src/parche_fd.py).

from senku.src.parche_fd import aplicar_parche
aplicar_parche()

from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner, get_environment
import time

get_environment().credits_stream = None
TIMEOUT = 60  # segundos por variante

filas_fd = []
for v in [1, 2, 3, 4, 5]:
    p = PDDLReader().parse_problem(
        str(RAIZ/'senku/pddl/dominio_senku.pddl'),
        str(RAIZ/f'senku/pddl/problemas/variante_{v}.pddl'))
    inicio = time.perf_counter()
    try:
        with OneshotPlanner(name='fast-downward') as planner:
            res = planner.solve(p, timeout=TIMEOUT)
        fila = {
            'variante': v,
            'estado': str(res.status).split('.')[-1],
            'movimientos': len(res.plan.actions) if res.plan else 0,
            'tiempo_s': round(time.perf_counter()-inicio, 2),
        }
    except Exception as e:
        fila = {
            'variante': v,
            'estado': f'ERROR: {type(e).__name__}',
            'movimientos': 0,
            'tiempo_s': round(time.perf_counter()-inicio, 2),
        }
    print(fila)
    filas_fd.append(fila)


## 5. BFS y Beam Search propios sobre los mismos PDDL

Cargamos cada problema PDDL con `unified_planning`, lo convertimos a nuestra representacion interna y aplicamos los dos algoritmos implementados a mano.

In [ ]:
import pandas as pd

filas = []
LIMITE_NODOS_BFS = 50_000
BETA = 200
INTENTOS = 5
ITER_MAX = 80

for numero, tablero in TABLEROS.items():
    ruta_problema = RAIZ / 'senku' / 'pddl' / 'problemas' / f'variante_{numero}.pddl'
    p = carga_problema_pddl(ruta_dominio, ruta_problema)
    pesos = pagoda_clasica(tablero)
    h_pag = heuristica_pagoda(p, pesos)
    h_com = heuristica_compuesta(p, pesos)

    r = busqueda_primero_anchura(p, limite_nodos=LIMITE_NODOS_BFS)
    filas.append({'variante': numero, 'algoritmo': 'BFS', 'heuristica': '-',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_pag, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'pagoda',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

    r = beam_search_con_reinicios(p, h_com, beta=BETA, intentos=INTENTOS, iteraciones_maximas=ITER_MAX)
    filas.append({'variante': numero, 'algoritmo': 'Beam', 'heuristica': 'compuesta',
                  'exito': r.exito, 'movimientos': len(r.movimientos),
                  'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})

df = pd.DataFrame(filas)
df

## 6. Influencia del parametro beta

Beam Search es incompleto y su capacidad de encontrar solucion depende mucho de la anchura del haz. Este experimento mide el efecto de beta sobre la profundidad alcanzada y el numero de exitos en multiples reinicios.

In [ ]:
VARIANTE_OBJETIVO = 1  # cruz inglesa
BETAS = [50, 100, 200, 500, 1000, 2000]
INTENTOS_BARRIDO = 5

p = ProblemaSenku.desde_tablero(TABLEROS[VARIANTE_OBJETIVO])
pesos = pagoda_clasica(p.tablero)
h = heuristica_compuesta(p, pesos)

filas_beta = []
for beta in BETAS:
    r = beam_search_con_reinicios(p, h, beta=beta, intentos=INTENTOS_BARRIDO, iteraciones_maximas=60)
    filas_beta.append({'beta': beta, 'exito': r.exito, 'mov': len(r.movimientos),
                       'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 3)})
pd.DataFrame(filas_beta)

## 7. Heuristica de conectividad (la decisiva)

La pagoda, pese a ser admisible, no contiene suficiente senal para guiar el haz hasta las (escasas) soluciones del Senku. La **heuristica de conectividad** ordena los estados por numero de componentes conexas de piezas (adyacencia ortogonal), penalizando piezas aisladas. La intuicion: para reducir el tablero a una sola pieza, el conjunto debe mantenerse cohesionado.

Con ella, beam search **si resuelve** la cruz inglesa, tanto en modo relajado (1 pieza en cualquier sitio) como estricto (1 pieza en el centro). El profesor ha aclarado que se da por valida cualquier solucion que deje una unica pieza, sin importar donde caiga.

In [ ]:
# Beam search + conectividad en modo relajado sobre las 5 variantes
filas_conect = []
config = {1: 300, 2: 800, 3: 1500, 4: 800, 5: 2000}
for numero, tablero in TABLEROS.items():
    p = ProblemaSenku.desde_tablero(tablero, modo_relajado=True)
    h = heuristica_conectividad(p)
    r = beam_search_con_reinicios(p, h, beta=config[numero], intentos=6, iteraciones_maximas=120)
    filas_conect.append({'variante': numero, 'casillas': len(tablero.casillas),
                         'beta': config[numero], 'exito': r.exito,
                         'movimientos': len(r.movimientos), 'piezas_finales': len(tablero.casillas)-1-len(r.movimientos) if r.exito else None,
                         'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 2)})
pd.DataFrame(filas_conect)

## 8. Estudio de la posicion del hueco inicial (cruz inglesa)

El profesor valora positivamente probar distintas disposiciones del hueco inicial y encontrar aquellas en las que es plausible terminar con la ultima pieza en el propio hueco (problema *complementario* del peg solitaire). Aqui usamos beam search + conectividad en modo estricto: para cada hueco candidato, la meta es terminar con una unica pieza en esa misma casilla.

Por simetria de la cruz inglesa basta examinar un cuadrante representativo.

In [ ]:
from senku.src.tableros import _tablero

base = TABLEROS[1]
candidatas = sorted({c for c in base.casillas if c[0] <= 3 and c[1] <= 3})
filas_huecos = []
for hueco in candidatas:
    t = _tablero(f'cruz_hueco_{hueco[0]}_{hueco[1]}', set(base.casillas), hueco=hueco, objetivo=hueco)
    p = ProblemaSenku.desde_tablero(t, modo_relajado=False)
    h = heuristica_conectividad(p)
    r = beam_search_con_reinicios(p, h, beta=500, intentos=8, iteraciones_maximas=120)
    filas_huecos.append({'hueco': hueco, 'complementario_ok': r.exito,
                         'movimientos': len(r.movimientos) if r.exito else None,
                         'nodos': r.nodos_expandidos, 'tiempo_s': round(r.tiempo_segundos, 1)})
pd.DataFrame(filas_huecos)

## 9. Conclusiones experimentales

1. **BFS no escala**: el espacio de estados crece exponencialmente y BFS no encuentra solucion en las variantes medianas con un presupuesto razonable de nodos.
2. **Fast Downward sirve como referencia**: resuelve V1 en 31 movimientos y demuestra la irresolubilidad estructural de V2 (5x5) y V4 (diamante).
3. **Beam Search con pagoda es incompleto**: aunque la heuristica es admisible, no contiene suficiente senal discriminativa; ni siquiera con beta=2000 y 20 reinicios resuelve V1.
4. **La heuristica de conectividad es la clave**: al ordenar por numero de componentes conexas, beam search resuelve la cruz inglesa en 31 movimientos (modo relajado y estricto) explorando solo unos miles de nodos.
5. **Estudio de huecos**: la posicion del hueco inicial determina si es plausible terminar en esa misma casilla. V3 y V5 no resuelven desde el centro pero si desde un brazo del tablero.